In [1]:
import pandas as pd

erp_customers = pd.DataFrame([
    [0, 'ABC Corp.', '123 Oak Ave', 'Montreal QC', 'CA'],
    [1, 'Contoso', 'Rue de Bell.', '75374 Paris', 'FR'],
    [2, 'DB Schenker', 'Kruppstr. 4', '45128 Essen', 'DE'],
    [3, 'Big Kahuna', '99 Ham Drive', 'Miami 33185', 'US'],
    [4, 'Schenker AG', 'Krupp Straße 4', 'Essen-City', 'DE']
], columns=['Id', 'CompanyName', 'AddressLine1', 'AddressLine2', 'Country']).set_index('Id')

crm_accounts = pd.DataFrame([
    ['a3b6l', 'Entity Hero', 'Bolzweg', '47839', 'Krefeld', 'DE'],
    ['ak44j', 'Luha Libre Ole', 'C. de Benova', '66667', 'Palencia', 'ES'],
    ['a89ci', 'Deutsche Bahn', 'Kruppstr. 4', '45128', 'Essen', 'DE'],
    ['aa341', 'Contoso Inc.', 'P.O. Box 123', '75374', 'Paris', 'FR'],
    ['a31bc', 'Bambini Pub', '66 Bobcat Lane', '09876', 'London', 'UK']
], columns=['AccId', 'Name', 'Street', 'ZipCode', 'City', 'Country']).set_index('AccId')

In [2]:
all_records = pd.concat([
    erp_customers.reset_index().assign(Source='erp'),
    (crm_accounts.reset_index().assign(Source='crm', AddressLine2=crm_accounts['ZipCode'] + ' ' + crm_accounts['City'])
     .rename(columns={'AccId': 'Id', 'Name': 'CompanyName', 'Street': 'AddressLine1'}))
], ignore_index=True).filter(items=['Source', 'Id', 'CompanyName', 'AddressLine1', 'AddressLine2', 'Country']).set_index('Id')

print(all_records)

      Source     CompanyName    AddressLine1 AddressLine2 Country
Id                                                               
0        erp       ABC Corp.     123 Oak Ave  Montreal QC      CA
1        erp         Contoso    Rue de Bell.  75374 Paris      FR
2        erp     DB Schenker     Kruppstr. 4  45128 Essen      DE
3        erp      Big Kahuna    99 Ham Drive  Miami 33185      US
4        erp     Schenker AG  Krupp Straße 4   Essen-City      DE
a3b6l    crm     Entity Hero         Bolzweg          NaN      DE
ak44j    crm  Luha Libre Ole    C. de Benova          NaN      ES
a89ci    crm   Deutsche Bahn     Kruppstr. 4          NaN      DE
aa341    crm    Contoso Inc.    P.O. Box 123          NaN      FR
a31bc    crm     Bambini Pub  66 Bobcat Lane          NaN      UK


In [ ]:
import recordlinkage as rl

# Linkage formulation:
indexer = rl.Index()
indexer.full()
candidate_pairs = indexer.index(erp_customers, crm_accounts)

comparer = rl.Compare()
comparer.string('CompanyName', 'Name', method='jarowinkler', label='name_similarity_score')
# Add more similarity functions here
scores = comparer.compute(candidate_pairs, erp_customers, crm_accounts)
print('Linkage scores:')
print(scores.head())

# Deduplication formulation:
indexer = rl.Index()
indexer.full()
candidate_pairs = indexer.index(all_records)

comparer = rl.Compare()
comparer.string('CompanyName', 'CompanyName', method='jarowinkler', label='name_similarity_score')
# Add more similarity functions here
scores = comparer.compute(candidate_pairs, all_records)
print('Deduplication scores:')
print(scores.head())